<a href="https://colab.research.google.com/github/GiovanniPioDelvecchio/NLP-Project/blob/GloVePreTrained/HumanValueRecognition_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [39]:
!pip install transformers
!pip install datasets

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [40]:
# Cell for the download of the datasets
!wget https://zenodo.org/record/7550385/files/arguments-training.tsv
!wget https://zenodo.org/record/7550385/files/labels-training.tsv
!wget https://zenodo.org/record/7550385/files/arguments-validation.tsv
!wget https://zenodo.org/record/7550385/files/labels-validation.tsv
!wget https://zenodo.org/record/7550385/files/arguments-test.tsv
!wget https://zenodo.org/record/7550385/files/arguments-validation-zhihu.tsv
!wget https://zenodo.org/record/7550385/files/labels-validation-zhihu.tsv

--2023-01-30 12:55:47--  https://zenodo.org/record/7550385/files/arguments-training.tsv
Resolving zenodo.org (zenodo.org)... 188.185.124.72
Connecting to zenodo.org (zenodo.org)|188.185.124.72|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1012498 (989K) [application/octet-stream]
Saving to: ‘arguments-training.tsv.2’

arguments-training. 100%[===================>] 988.77K   571KB/s    in 1.7s    

2023-01-30 12:55:50 (571 KB/s) - ‘arguments-training.tsv.2’ saved [1012498/1012498]

--2023-01-30 12:55:51--  https://zenodo.org/record/7550385/files/labels-training.tsv
Resolving zenodo.org (zenodo.org)... 188.185.124.72
Connecting to zenodo.org (zenodo.org)|188.185.124.72|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 253843 (248K) [application/octet-stream]
Saving to: ‘labels-training.tsv.2’

labels-training.tsv 100%[===================>] 247.89K   564KB/s    in 0.4s    

2023-01-30 12:55:53 (564 KB/s) - ‘labels-training.tsv.2’ saved [

In [41]:
# imports for dataset loading
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# torch imports
import torch
import torchtext
from torchtext.data import get_tokenizer
from torchtext.vocab import GloVe
from torch.utils.data import DataLoader
from torchtext.data.functional import to_map_style_dataset
from torch import nn
from torch.nn import functional as F
from torch.optim import Adam

# progress bar
from tqdm import tqdm
# garbage collector
import gc

In [42]:
seed = 10
torch.use_deterministic_algorithms(True)

In [43]:
def huggingface_from_pandas(pandas_df):
  hf_ds = Dataset.from_pandas(pandas_df, preserve_index=False)
  hf_ds = hf_ds.remove_columns(["Argument ID", "Argument ID2"])
  hf_ds = hf_ds.map(lambda x:{"labels": [int(x[col]) for col in hf_ds.column_names if
                                      col not in ['Conclusion', 'Stance', 'Premise']]})
  label_cols = [col for col in hf_ds.column_names if col not in ['Conclusion', 'Stance', 'Premise', "labels"]]
  hf_ds = hf_ds.remove_columns(label_cols)
  return hf_ds, label_cols

In [44]:
# Dataset loading and splitting
raw_training = pd.read_csv("arguments-training.tsv", encoding='utf-8', sep='\t', header=0)
raw_training_lab = pd.read_csv("labels-training.tsv", encoding='utf-8', sep='\t', header=0)
raw_test = pd.read_csv("arguments-validation.tsv", encoding='utf-8', sep='\t', header=0)
raw_test_lab = pd.read_csv("labels-validation.tsv", encoding='utf-8', sep='\t', header=0)

train = raw_training.join(raw_training_lab,how='inner' ,lsuffix='2') # joining labels
test = raw_test.join(raw_test_lab, how='inner', lsuffix='2') # joining labels
train, val = train_test_split(train ,train_size=.80, random_state=seed) # splitting training

train_ds, label_list = huggingface_from_pandas(train)
val_ds, _ = huggingface_from_pandas(val)
test_ds, _ = huggingface_from_pandas(test)

print(train_ds[0])
print(label_list)

whole_dataset = DatasetDict()
whole_dataset["train"] = train_ds.with_format("torch")
whole_dataset["val"] = val_ds.with_format("torch")
whole_dataset["test"] = test_ds.with_format("torch")

  0%|          | 0/4314 [00:00<?, ?ex/s]

  0%|          | 0/1079 [00:00<?, ?ex/s]

  0%|          | 0/1896 [00:00<?, ?ex/s]

{'Conclusion': 'We should ban the Church of Scientology', 'Stance': 'in favor of', 'Premise': "Scientology is not a true religion it is a sect or a cult which brainwashes it's followers and makes money from them.", 'labels': [1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0]}
['Self-direction: thought', 'Self-direction: action', 'Stimulation', 'Hedonism', 'Achievement', 'Power: dominance', 'Power: resources', 'Face', 'Security: personal', 'Security: societal', 'Tradition', 'Conformity: rules', 'Conformity: interpersonal', 'Humility', 'Benevolence: caring', 'Benevolence: dependability', 'Universalism: concern', 'Universalism: nature', 'Universalism: tolerance', 'Universalism: objectivity']


In [7]:
print(whole_dataset.keys())
print(whole_dataset["train"])

dict_keys(['train', 'val', 'test'])
Dataset({
    features: ['Conclusion', 'Stance', 'Premise', 'labels'],
    num_rows: 4314
})


In [8]:
print(whole_dataset["train"]["labels"][0].shape)
print(whole_dataset["train"]["labels"])

torch.Size([20])
tensor([[1, 1, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 1],
        ...,
        [0, 0, 0,  ..., 0, 0, 1],
        [0, 0, 0,  ..., 0, 1, 0],
        [0, 0, 1,  ..., 0, 0, 1]])


In [45]:
# Pretrained GloVe setup

global_vectors = GloVe(name='6B', dim=100)

# the current choice is to give an id to each word
tokenizer = get_tokenizer("basic_english")

embeddings = global_vectors.get_vecs_by_tokens(tokenizer("Hello, How are you?"), lower_case_backup=True)

print(embeddings.shape)

torch.Size([6, 100])


In [46]:
max_words = 25
embed_len = 100
batch_size = 20

# collate function where the Premises are tokenized and embedded in batches
def vectorize_batch(batch):
    X = [elem["Premise"] for elem in batch]
    Y = [elem["labels"] for elem in batch]
    X = [tokenizer(x[0]) for x in X]
    X = [tokens+[""] * (max_words-len(tokens))  if len(tokens)<max_words else tokens[:max_words] for tokens in X]
    X_tensor = torch.zeros(len(batch), max_words, embed_len)
    Y_tensor = torch.zeros(len(batch), Y[0].shape[0])
    for i, tokens in enumerate(X):
        X_tensor[i] = global_vectors.get_vecs_by_tokens(tokens)
        Y_tensor[i] = Y[i]
    X_to_return = X_tensor.reshape(len(batch), -1)
    Y_to_return = Y_tensor.reshape(len(batch), -1)

    #print(X_to_return)
    #print("____________")
    #print(Y_to_return)

    return X_to_return, Y_to_return

train_dataset = whole_dataset["train"].remove_columns(["Stance", "Conclusion"])
val_dataset = whole_dataset["val"].remove_columns(["Stance", "Conclusion"])
print(val_dataset.shape)

# Construction of the Dataloaders for train and validation
train_loader = DataLoader(train_dataset, batch_size=batch_size, collate_fn=vectorize_batch)
val_loader  = DataLoader(val_dataset, batch_size=batch_size, collate_fn=vectorize_batch)

(1079, 2)


In [107]:
num_classes = 20

# Simple model to perform some tests with pytorch
class EmbeddingClassifier(nn.Module):
    def __init__(self):
        super(EmbeddingClassifier, self).__init__() 
        # Not sure about this
        #self.seq_length = batch_size
        self.input_dim = max_words*embed_len
        self.sequence_length = 20 
        self.num_layers = 1
        self.gru_output_dim = self.input_dim
        self.gru = nn.GRU(input_size = self.input_dim,
                          hidden_size = self.input_dim,
                          num_layers = 1, 
                          bidirectional = False)

        self.linear_1 = nn.Linear(max_words*embed_len, 256)
        self.relu = nn.ReLU()

        self.linear_2 = nn.Linear(256,128)
        #nn.ReLU(),

        self.linear_3 = nn.Linear(128,64)
        #nn.ReLU(),

        self.linear_4 = nn.Linear(64, num_classes)
        
                

    def forward(self, X_batch):
        h0 = torch.zeros(1, self.gru_output_dim)
        out, hn = self.gru(X_batch, h0)
        out = self.linear_1(out)
        out = self.relu(out)
        out = self.linear_2(out)
        out = self.relu(out)
        out = self.linear_3(out)
        out = self.relu(out)
        out = self.linear_4(out)
        return out

# Function needed to compute the validation loss and the accuracy
def CalcValLossAndAccuracy(model, loss_fn, val_loader):
    with torch.no_grad():
        Y_shuffled, Y_preds, losses = [],[],[]
        for X, Y in val_loader:
            preds = model(X)
            loss = loss_fn(preds, Y)
            losses.append(loss.item())

            Y_shuffled.append(Y)
            Y_preds.append(preds.argmax(dim=-1))

        Y_shuffled = torch.cat(Y_shuffled)
        Y_preds = torch.cat(Y_preds)

        print("Valid Loss : {:.3f}".format(torch.tensor(losses).mean()))
        print("Valid Acc  : {:.3f}".format(accuracy_score(Y_shuffled.detach().numpy(), Y_preds.detach().numpy())))

# Training function
def TrainModel(model, loss_fn, optimizer, train_loader, val_loader, epochs=10):
    for i in range(1, epochs+1):
        losses = []
        for X, Y in tqdm(train_loader):
            Y_preds = model(X)

            loss = loss_fn(Y_preds, Y)
            losses.append(loss.item())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if i%5==0:
            print("Train Loss : {:.3f}".format(torch.tensor(losses).mean()))
            CalcValLossAndAccuracy(model, loss_fn, val_loader)

In [108]:
epochs = 3
learning_rate = 1e-3

loss_fn = nn.BCEWithLogitsLoss()
embed_classifier = EmbeddingClassifier()
optimizer = Adam(embed_classifier.parameters(), lr=learning_rate)

TrainModel(embed_classifier, loss_fn, optimizer, train_loader, val_loader, epochs)

100%|██████████| 216/216 [04:29<00:00,  1.25s/it]


In [120]:
def MakePredictions(model, loader):
    Y_shuffled, Y_preds = [], []
    for X, Y in loader:
        preds = model(X)
        Y_preds.append(preds)
        #Y_shuffled.append(Y)
    gc.collect()
    Y_preds = torch.cat(Y_preds)
    Y_preds = Y_preds.sigmoid()
    return Y_preds.detach()

Y_preds = MakePredictions(embed_classifier, val_loader)
print(Y_preds)

tensor([[0.2304, 0.3178, 0.0431,  ..., 0.0629, 0.1042, 0.1763],
        [0.2069, 0.2867, 0.0573,  ..., 0.0768, 0.1103, 0.1934],
        [0.1724, 0.2391, 0.0603,  ..., 0.0764, 0.1108, 0.2278],
        ...,
        [0.2212, 0.2954, 0.0631,  ..., 0.0838, 0.1185, 0.2004],
        [0.1919, 0.3014, 0.0695,  ..., 0.0971, 0.1230, 0.1973],
        [0.2468, 0.3051, 0.0655,  ..., 0.0874, 0.1258, 0.2044]])


In [116]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

print(len(Y_preds))
print(len(Y_preds[0]))
print(np.argmax(Y_preds[0]))
print(Y_preds[0])

1079
20
tensor(8)
tensor([0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.])


In [142]:
def keep_above_thresh(Y_preds, thr):
  Y_preds_thr = np.copy(Y_preds.numpy())
  max_rows = Y_preds_thr.shape[0]
  max_cols = Y_preds_thr.shape[1]
  for i in range(max_rows):
    new_row = np.array([1 if Y_preds_thr[i][j] > thr else 0 for j in range(max_cols)])
    Y_preds_thr[i] = new_row
  return Y_preds_thr

Y_preds_thr = keep_above_thresh(Y_preds, 0.30)
print(Y_preds[0])
print(Y_preds_thr[0])

tensor([0.2304, 0.3178, 0.0431, 0.0208, 0.3170, 0.1145, 0.1126, 0.0564, 0.4410,
        0.2118, 0.1156, 0.1666, 0.0180, 0.0537, 0.2191, 0.1677, 0.3555, 0.0629,
        0.1042, 0.1763])
[0. 1. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]


In [143]:
from sklearn.metrics import f1_score
def compute_f1_macro(M_true, M_pred):
    f1_scores = []
    for i in range(M_true.shape[1]):
        true = M_true[:, i]
        pred = M_pred[:, i]
        f1_scores.append(f1_score(true, pred, average='macro'))
    return np.mean(f1_scores)

Y_true = val_dataset["labels"]
f1_macro = compute_f1_macro(Y_true, Y_preds_thr)
print(f1_macro)

0.4543779450711062


In [144]:
entry_to_check = 0
print(np.array(Y_true[entry_to_check].numpy(), dtype="float64"))
print(Y_preds_thr[entry_to_check])

[1. 0. 0. 0. 0. 0. 0. 0. 1. 1. 0. 0. 0. 0. 0. 1. 0. 1. 0. 0.]
[0. 1. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]


In [125]:
from sklearn.metrics import classification_report

Y_true = val_dataset["labels"]
Y_preds_np = Y_preds.numpy()
Y_true_cl = np.argmax(Y_true, axis = 1)
Y_preds_cl = np.argmax(Y_preds_np, axis = 1)


print(classification_report(Y_true_cl, Y_preds_cl, zero_division = 0))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       190
           1       0.00      0.00      0.00       175
           2       0.00      0.00      0.00        16
           3       0.00      0.00      0.00         3
           4       0.00      0.00      0.00       172
           5       0.00      0.00      0.00        60
           6       0.00      0.00      0.00        34
           7       0.00      0.00      0.00        17
           8       0.14      0.64      0.23       161
           9       0.00      0.00      0.00       120
          10       0.00      0.00      0.00        18
          11       0.00      0.00      0.00        34
          12       0.00      0.00      0.00         5
          13       0.00      0.00      0.00        12
          14       0.00      0.00      0.00        22
          15       0.00      0.00      0.00         7
          16       0.02      0.33      0.03        18
          17       0.00    